# 📘 Chapter 08 — Advanced Gradient Boosting (LightGBM)

## 🎯 Objectives
In this chapter, we will implement **LightGBM**, a highly efficient gradient boosting model.

To handle the large dataset (~1GB CSV) on a 16GB RAM machine, we will first convert the data to **Parquet** format using a chunking strategy.

We will then train the model and compare its performance with the Random Forest model from previous chapters.

### 🧩 Step 01 — Memory-Efficient Data Conversion
We use chunking to read the CSV in small parts and append them to a Parquet file. 

This keeps memory usage low and ensures **100% data integrity**.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import gc
import time
import sys
import os

from src.config import PROCESSED_DIR
from src.utils.emoji_log import task, success, info

# Setting paths
CSV_PATH = PROCESSED_DIR / "Taiwan.csv"
PARQUET_PATH = PROCESSED_DIR / "full_data.parquet"

print(f"📂 來源檔案: {CSV_PATH}")
print(f"📂 目標檔案: {PARQUET_PATH}")

📂 來源檔案: C:\Users\dinni\OneDrive\桌面\air_pollution\data\processed\Taiwan.csv
📂 目標檔案: C:\Users\dinni\OneDrive\桌面\air_pollution\data\processed\full_data.parquet


In [2]:
convert_csv_to_parquet(CSV_PATH, PARQUET_PATH)

🚀 Start converting: Taiwan.csv -> full_data.parquet...
💬 Chunk Size: 100000
💬 Processed 1000000 rows...
💬 Processed 2000000 rows...
💬 Processed 3000000 rows...
💬 Processed 4000000 rows...
💬 Processed 5000000 rows...
🏁 Done! Total rows: 5823862
💬 Time elapsed: 28.44 seconds
💬 New file size: 97.45 MB


In [3]:
df = pd.read_parquet(PARQUET_PATH)
df.head()

,date,sitename,county,aqi,status,so2,co,o3,o3_8hr,pm10,...,winddirec,co_8hr,longitude,latitude,year,month,day,weekday,hour,season
0,2024-08-31 23:00:00,Hukou,Hsinchu_County,62.0,Moderate,0.9,0.17,35.0,40.2,18.0,...,225.0,0.2,121.038869,24.900097,2024,8,31,5,23,Summer
1,2024-08-31 23:00:00,Zhongming,Taichung_City,50.0,Good,1.6,0.32,27.9,35.1,27.0,...,184.0,0.2,120.641092,24.151958,2024,8,31,5,23,Summer
2,2024-08-31 23:00:00,Zhudong,Hsinchu_County,45.0,Good,0.4,0.17,25.1,40.6,21.0,...,210.0,0.2,121.088955,24.740914,2024,8,31,5,23,Summer
3,2024-08-31 23:00:00,Hsinchu,Hsinchu_City,42.0,Good,0.8,0.20,30.0,35.9,19.0,...,239.0,0.2,120.972368,24.805636,2024,8,31,5,23,Summer
4,2024-08-31 23:00:00,Toufen,Miaoli_County,50.0,Good,1.0,0.16,33.5,35.9,18.0,...,259.0,0.1,120.898693,24.696907,2024,8,31,5,23,Summer


### 🧩 Step 02 — Load Data & Train-Test Split
Load the converted Parquet file and split the data into training and testing sets (80/20 split).

In [7]:
from src.features.feature_engineering import (
    add_rolling_features,
    clip_pollutants,
    handle_outliers_iqr,
    log_transform_features,
    scale_features,
)
from src.modeling.train_baseline import build_features, split_train_test

# 1. Load Parquet
task("Loading data from Parquet...")
start = time.time()
df = pd.read_parquet(PARQUET_PATH)
success(f"Loaded {df.shape[0]} rows in {time.time() - start:.2f} seconds!")

# 2. Feature Engineering Pipeline (following main_modeling.py flow)
task("Starting Feature Engineering...")

# Step 1: Clip pollutants
df = clip_pollutants(df)

# Step 2: Rolling features (3d, 7d)
df = add_rolling_features(df)

# Step 3: IQR Outlier handling
df = handle_outliers_iqr(df)

# Step 4: Log transform
df = log_transform_features(df)

success("Feature Engineering Complete!")

# 3. Build X, y
X, y = build_features(df)

# 4. Split into train/test (80/20)
data_split = split_train_test(X, y)
X_train, X_test = data_split["X_train"], data_split["X_test"]
y_train, y_test = data_split["y_train"], data_split["y_test"]

# 5. Scaling (using training set scaler)
X_train_scaled, scaler = scale_features(X_train, mode="train", save_=False)
X_test_scaled = scale_features(X_test, scaler=scaler, mode="test")

info(f"Train shape: {X_train_scaled.shape}")
info(f"Test shape: {X_test_scaled.shape}")

# Release memory
del df, X, y, X_train, X_test
gc.collect()

🚀 Loading data from Parquet...
✅ Loaded 5823862 rows in 2.05 seconds!
🚀 Starting Feature Engineering...
✅ The pollutants limit has been set.
⚠️ Skipping nox due to NO + NO2 already exist.
✅ Rolling features added.
✅ IQR has been set.
✅ Pollutants skewes has been smoothed.
✅ Feature Engineering Complete!
💬 Features shape: (5823862, 32), Target shape: (5823862,)
✅ Data successfully split! Train: (4659089, 32), Test: (1164773, 32)
💬 Train shape: (4659089, 32)
💬 Test shape: (1164773, 32)


638

### 🚀 Step 03 — LightGBM Modeling
Train a LightGBM Regressor with early stopping to prevent overfitting.

In [ ]:
from src.modeling.evaluate_model import evaluate_and_save

# 1. Setting LightGBM params
# These parameters are for regression tasks and optimized for large datasets
params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "n_estimators": 5000,  # Maximum number of trees (will be truncated by early_stopping)
    "learning_rate": 0.05,  # Learning rate
    "num_leaves": 63,  # Number of leaves (controls model complexity)
    "max_depth": -1,  # Tree depth (-1 means unlimited)
    "subsample": 0.8,  # Random sampling ratio (prevents overfitting)
    "colsample_bytree": 0.8,  # Feature sampling ratio
    "random_state": 42,
    "n_jobs": -1,  # Use all CPU cores
    "device": "cpu",  # If you have GPU, change to "gpu"
}

task("Training LightGBM model...")
start = time.time()

# 2. Build model
model = lgb.LGBMRegressor(**params)

# 3. Train model (with Early Stopping)
model.fit(
    X_train_scaled,
    y_train,
    eval_set=[(X_test_scaled, y_test)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)],
)

elapsed = time.time() - start
success(f"Training finished in {elapsed:.2f} seconds!")

### 📊 Step 04 — Evaluation & Comparison
Evaluate the model using MAE, RMSE, and R², and compare the results with the Random Forest baseline.